In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")
# groq_api_key

In [2]:
from langchain_groq import ChatGroq
model=ChatGroq(model="qwen/qwen3.6-27b", groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}}, client=<groq.resources.chat.completions.Completions object at 0x000001AB5AE46050>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001AB5AE45F60>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage, SystemMessage
model.invoke([HumanMessage(content="Hi, I am raghvendra, I am an AI Engineer")])

AIMessage(content="\n<think>\nHere's a thinking process:\n\n1.  **Analyze User Input:**\n   - **Name:** Raghvendra\n   - **Profession:** AI Engineer\n   - **Tone:** Greeting/Introduction\n\n2.  **Identify Key Elements:**\n   - The user is introducing themselves professionally.\n   - They've shared their name and role.\n   - No specific question or request is made yet.\n\n3.  **Determine Response Goals:**\n   - Acknowledge the introduction warmly\n   - Use their name to personalize the response\n   - Acknowledge their profession (AI Engineer)\n   - Offer assistance relevant to their field\n   - Keep it open-ended to encourage further conversation\n\n4.  **Draft Response (Mental Refinement):**\n   Hi Raghvendra! Nice to meet you. It's great to connect with an AI Engineer. How can I assist you today? Whether you're working on model training, MLOps, prompt engineering, architecture design, or just want to brainstorm ideas, I'm here to help. What's on your mind?\n\n5.  **Check Against Guide

In [4]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi, I am raghvendra, I am an AI Engineer"),
        AIMessage(content="Hi Raghvendra! Nice to meet you. It's great to connect with an AI Engineer. How can I assist you today? Whether you're working on model training, MLOps, prompt engineering, architecture design, or just want to brainstorm ideas, I'm here to help. What's on your mind?. "),
        HumanMessage(content="What is my Name, and What do i do?")

    ]
)

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "What is my Name, and What do i do?"\n   - This refers back to the previous message where they introduced themselves: "Hi, I am raghvendra, I am an AI Engineer"\n\n2.  **Identify Key Information:**\n   - Name: Raghvendra\n   - Profession/Role: AI Engineer\n\n3.  **Formulate Response:**\n   - Acknowledge the question directly\n   - State the name clearly\n   - State the profession/role clearly\n   - Keep it concise and friendly\n   - Optionally add a polite follow-up\n\n   Draft: "Your name is Raghvendra, and you work as an AI Engineer. Let me know how I can assist you with your projects or any AI-related questions you might have!"\n\n4.  **Self-Correction/Verification:**\n   - Does it match the user\'s input? Yes.\n   - Is it accurate? Yes.\n   - Is the tone appropriate? Yes, professional and helpful.\n   - No extra fluff needed.\n\n   Final check matches the draft. Proceed.✅\n</th

# Message History

We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(sesssion_id:str)->BaseChatMessageHistory:
    if sesssion_id not in store:
        store[sesssion_id]=ChatMessageHistory()
    return store[sesssion_id]

with_message_history=RunnableWithMessageHistory(model, get_session_history) # Connects our model with 
# chat history

e:\LangChain\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [8]:
config={
    "configurable":{
        "session_id":"chat1"
    }
}

In [9]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Raghvendra and I am a AI Engineer")],
    config=config
)

In [11]:
response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Hi, My name is Raghvendra and I am a AI Engineer"\n   - Key information: Name = Raghvendra, Profession = AI Engineer\n   - Tone: Friendly, introductory\n   - Intent: Greeting and self-introduction, likely expecting a response that acknowledges this and opens the door for collaboration/conversation.\n\n2.  **Identify Key Requirements for Response:**\n   - Acknowledge the name and profession\n   - Match the friendly/professional tone\n   - Offer assistance relevant to an AI Engineer\n   - Keep it concise and open-ended\n\n3.  **Draft - Mental Refinement:**\n   "Hi Raghvendra! Great to meet you. It\'s awesome that you\'re an AI Engineer—there\'s so much exciting work happening in the field right now. How can I assist you today? Whether it\'s brainstorming model architectures, debugging training pipelines, exploring new papers, or just chatting about AI trends, I\'m here to help!"\n\n4.  **Check Agains

In [12]:
response=with_message_history.invoke(
    [HumanMessage(content="What is my Name")],
    config=config
)

In [13]:
response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "What is my Name"\n2.  **Check Context/History:** In the previous turn, the user explicitly stated: "Hi, My name is Raghvendra and I am a AI Engineer"\n3.  **Extract Information:** The user\'s name is Raghvendra.\n4.  **Formulate Response:** Acknowledge the name clearly and politely. Keep it concise.\n5.  **Draft Response:** Your name is Raghvendra! Is there anything specific you\'d like help with today, Raghvendra?\n6.  **Self-Correction/Verification:** Matches the context exactly. No extra fluff. Direct and accurate. Ready.✅\n</think>\n\nYour name is **Raghvendra**! How can I assist you today?'

In [14]:
config1={
    "configurable":{
        "session_id":"chat1"
    }
}
response=with_message_history.invoke(
    [HumanMessage(content="What is my Name")],
    config=config1
)

In [15]:
response.text

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "What is my Name" again.\n2.  **Recall Context:** In the first message, the user explicitly stated: "Hi, My name is Raghvendra and I am a AI Engineer". I acknowledged it. In the second turn, I correctly answered "Your name is Raghvendra!". Now they\'re asking the exact same question again.\n3.  **Identify Potential Intent:** \n   - Testing memory/context retention\n   - Simple repetition/check\n   - Might be a follow-up to see if I\'m consistent\n   - Could be a test of the system\'s statefulness\n4.  **Formulate Response:** \n   - Keep it consistent and direct\n   - Reiterate the name clearly\n   - Maintain a friendly, professional tone\n   - Optionally acknowledge the repetition lightly if needed, but keep it straightforward\n   - Example: "Your name is Raghvendra, as you mentioned earlier. Let me know how I can assist you!"\n5.  **Self-Correction/Refinement during thought:** \n   - No need to overco

In [16]:
config1={
    "configurable":{
        "session_id":"chat2"
    }
}
response=with_message_history.invoke(
    [HumanMessage(content="What is my Name")],
    config=config1
)

In [17]:
response.text

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "What is my Name"\n2.  **Identify Key Constraint:** I am an AI assistant. I don\'t have access to personal information about users unless they explicitly provide it in the current conversation or it\'s stored in a persistent memory system (which, in this context, I don\'t have access to).\n3.  **Determine Response Strategy:** \n   - Acknowledge the question\n   - Explain that I don\'t know their name\n   - Clarify that I don\'t have access to personal data\n   - Offer to remember it if they tell me\n   - Keep it polite and helpful\n4.  **Draft Response (Mental):** I don\'t know your name! I don\'t have access to personal information or past conversations. If you\'d like, you can tell me your name and I\'ll be happy to use it in our chat.\n5.  **Refine Response:** Check for accuracy, tone, and alignment with AI capabilities. The draft is clear, honest, and helpful. It matches standard AI behavior guidel

# Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [39]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistent, Answer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

parser=StrOutputParser()

chain=prompt|model|parser

In [24]:
chain.invoke({"messages":[
    HumanMessage(
        content="Hi My name s Raghvendra"
    )
]})

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Hi My name s Raghvendra"\n   - There\'s a typo ("s" instead of "is"), but the meaning is clear: The user is introducing themselves as Raghvendra.\n   - This is a simple greeting/introduction.\n\n2.  **Identify Key Requirements:**\n   - Acknowledge the greeting\n   - Use their name (Raghvendra)\n   - Respond politely and warmly\n   - Offer assistance\n\n3.  **Formulate Response (Mental):**\n   - "Hi Raghvendra! Nice to meet you. How can I help you today?"\n   - Check tone: Friendly, professional, helpful.\n   - Check accuracy: Uses correct name, acknowledges introduction.\n   - Keep it concise.\n\n4.  **Refine Response:**\n   - "Hello Raghvendra! Nice to meet you. How can I assist you today?"\n   - This matches the mental formulation and meets all requirements.\n\n5.  **Final Output Generation:** (matches the refined response)✅\n   - Ready. Output matches the final refined version.

In [25]:
with_message_history=RunnableWithMessageHistory(
    chain,get_session_history
)

e:\LangChain\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [28]:
config={
    "configurable":{
        "session_id":"chat3"
    }
}
response=with_message_history.invoke(
    [HumanMessage(content="My Name is Raghvendra")],
    config=config
)

In [29]:
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input**: The user says "My Name is Raghvendra"\n2.  **Identify Key Information**: The user\'s name is Raghvendra.\n3.  **Determine Response Goal**: Acknowledge the name, greet the user by name, and maintain a friendly, helpful tone.\n4.  **Formulate Response**: \n   - Acknowledge: "Nice to meet you, Raghvendra!"\n   - Offer assistance: "How can I help you today?"\n   - Keep it concise and natural.\n5.  **Check Constraints**: The prompt says "Answer all the question to the nest of your ability" (typo: "nest" -> "best"). I\'ll keep it helpful and accurate.\n6.  **Draft Response**: "Nice to meet you, Raghvendra! How can I assist you today?"\n7.  **Final Polish**: Matches the draft, friendly, acknowledges the name, invites next question. Ready.✅\n</think>\n\nNice to meet you, Raghvendra! How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 240, 'prompt_tokens

In [49]:
# Add More Complexity
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistent, Answer all the question to the nest of your ability in {language}"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model|parser

In [51]:
response=chain.invoke({
    "messages":[HumanMessage(content="My Name is Raghvendra")],
    "language":"Hindi"
})

response

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "My Name is Raghvendra"\n   - Language: English\n   - Context: Simple introduction/statement of name\n   - Constraint from system prompt: "Answer all the question to the nest of your ability in Hindi" (Note: "nest" is likely a typo for "best", but the constraint is clear: respond in Hindi)\n\n2.  **Identify Key Requirements:**\n   - Acknowledge the name\n   - Respond in Hindi\n   - Be polite and helpful\n   - Follow the constraint strictly\n\n3.  **Formulate Response (Mental Draft in Hindi):**\n   - Namaste Raghvendra ji! Aapka naam jaan kar khushi hui. Main aapki kaise madad kar sakta hoon?\n   - Check: Is it in Hindi? Yes. Does it acknowledge the name? Yes. Is it polite and helpful? Yes.\n\n4.  **Refine Response (Hindi):**\n   नमस्ते राघवेन्द्र जी! आपका परिचय हूँ। मैं आपकी कैसे सहायता कर सकता हूँ? यदि आपके कोई प्रश्न हैं या किसी विषय पर चर्चा करनी है, तो बेझिझक पूछें।\n\n   (Note: Fixed grammar sl

In [52]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

e:\LangChain\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [53]:
config={
    "configurable":{
        "session_id":"chat4"
    }
}
response=with_message_history.invoke(
    {
    "messages":[HumanMessage(content="My Name is Raghvendra")],
    "language":"Hindi"
},
    config=config
)

In [54]:
response

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "My Name is Raghvendra" (repeated multiple times)\n   - Previous interactions show I\'ve already acknowledged the name in Hindi and English.\n   - The user is repeating the same statement.\n\n2.  **Identify Key Constraints:**\n   - System prompt: "Answer all the question to the nest of your ability in Hindi" (Note: "nest" is likely a typo for "best", but the instruction is clear: respond in Hindi).\n   - I need to maintain consistency with the Hindi language requirement.\n\n3.  **Determine Response Strategy:**\n   - Acknowledge the name again politely in Hindi.\n   - Keep it concise since it\'s a repetition.\n   - Offer assistance in Hindi as per instructions.\n   - Maintain a respectful and helpful tone.\n\n4.  **Draft Response (Mental Refinement in Hindi):**\n   नमस्ते राघवेन्द्र जी! आपका नाम मैंने नोट कर लिया है। बताइए, आज मैं आपकी किस प्रकार सहायता कर सकता हूँ?\n\n5.  **Check Against Constraints

In [55]:
# 2. Extract raw text from the AIMessage object
raw_text = response

import re
# 3. Strip out the <think>...</think> block generated by models like Qwen
clean_text = re.sub(r'<think>.*?</think>', '', raw_text, flags=re.DOTALL).strip()

# 4. Print only the clean output
print(clean_text)

नमस्ते राघवेन्द्र जी! आपका नाम मैंने ध्यान से नोट कर लिया है। बताइए, आज मैं आपकी किस प्रकार सहायता कर सकता हूँ?


# Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [64]:
from langchain_core.messages import SystemMessage, trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter="approximate",
    include_system=True,
    allow_partial=False,
    start_on="human"
)

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [66]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
)

chain.invoke({
    "messages": messages+[HumanMessage(content="What ice cream do i like")],
    "language":"English"
}
)

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "What ice cream do i like"\n   - This is a personal question about the user\'s preferences.\n   - As an AI, I don\'t have access to personal information about users unless they\'ve shared it in the current conversation or it\'s stored in a memory/context system (which I don\'t have access to in this stateless interaction).\n\n2.  **Identify Key Constraints:**\n   - I don\'t know the user\'s personal preferences.\n   - I must be honest about my limitations.\n   - I should respond politely and offer to help if they tell me their favorite or want recommendations.\n\n3.  **Formulate Response:**\n   - Acknowledge the question\n   - State clearly that I don\'t have access to personal information\n   - Ask them to share their favorite or offer to help them find one they might like\n   - Keep it friendly and conversational\n\n   Draft: I don’t actually know what ice cream you like since I 

In [68]:
response

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "My Name is Raghvendra" (repeated multiple times)\n   - Previous interactions show I\'ve already acknowledged the name in Hindi and English.\n   - The user is repeating the same statement.\n\n2.  **Identify Key Constraints:**\n   - System prompt: "Answer all the question to the nest of your ability in Hindi" (Note: "nest" is likely a typo for "best", but the instruction is clear: respond in Hindi).\n   - I need to maintain consistency with the Hindi language requirement.\n\n3.  **Determine Response Strategy:**\n   - Acknowledge the name again politely in Hindi.\n   - Keep it concise since it\'s a repetition.\n   - Offer assistance in Hindi as per instructions.\n   - Maintain a respectful and helpful tone.\n\n4.  **Draft Response (Mental Refinement in Hindi):**\n   नमस्ते राघवेन्द्र जी! आपका नाम मैंने नोट कर लिया है। बताइए, आज मैं आपकी किस प्रकार सहायता कर सकता हूँ?\n\n5.  **Check Against Constraints